# 🎙 Vlastní hlas pro Překladač (Piper) – trénování zdarma v Google Colab

Z vašich nahrávek (ZIP z **Překladače → Nastavení → Součásti → Vlastní hlas → Nahrávací studio → Export pro trénování**)
se dotrénuje český hlas **Piper** z hlasu Jirka. Výsledek (`.onnx` a `.onnx.json`) pak v Překladači importujete
a hlas čte knihy i audioknihy **bez internetu**.

**Než začnete:** nahoře *Runtime / Běhové prostředí → Změnit typ běhového prostředí → GPU (T4)*.
Pak spouštějte buňky postupně (▶). Trénování trvá dlouho (u hodiny nahrávek řádově desítky hodin na bezplatné grafice T4) – bezplatný Colab
se po několika hodinách odpojí, průběh je ale na vašem Google Disku a buňkami 1–5 se pokračuje dál.
Rychlejší je Colab Pro nebo počítač s grafikou NVIDIA. Poslechnout si mezivýsledek jde kdykoli buňkou 6.

> Tento sešit používá oficiální trénování Piper (projekt *OHF-Voice/piper1-gpl*). Když se něco změní v jejich
> instalaci, chybu napíše buňka 2 – pošlete ji a sešit se upraví.

In [2]:
# 1) Google Disk – sem se ukládá průběh trénování a hotový hlas
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/Prekladac-vlastni-hlas'
os.makedirs(WORK, exist_ok=True)
print('Pracovní složka:', WORK)

Mounted at /content/drive
Pracovní složka: /content/drive/MyDrive/Prekladac-vlastni-hlas


In [3]:
# 2) Instalace trénování Piper (asi 5 minut)
!apt-get -qq install -y espeak-ng build-essential cmake ninja-build > /dev/null
!git clone -q https://github.com/OHF-Voice/piper1-gpl.git /content/piper1-gpl || true
%cd /content/piper1-gpl
!pip -q install -e '.[train]'
!./build_monotonic_align.sh
!python3 setup.py build_ext --inplace > /dev/null
%cd /content

/content/piper1-gpl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for piper-tts (pyproject.toml) ... done
Compiling /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx because it changed.
[1/1] Cythonizing /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
/usr/local/lib/python3.13/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
performance hint: core.pyx:7:5: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_each' as 'noexcept' if you control the definition an

In [ ]:
# 3) Nahrajte ZIP s nahrávkami z Překladače (nebo ho dejte do složky Prekladac-vlastni-hlas na Disku)
import glob, zipfile, os
zips = glob.glob(WORK + '/*.zip')
if not zips:
    from google.colab import files
    up = files.upload()
    for name, data in up.items():
        open(os.path.join(WORK, name), 'wb').write(data)
    zips = glob.glob(WORK + '/*.zip')
ZIP = sorted(zips, key=os.path.getmtime)[-1]
zipfile.ZipFile(ZIP).extractall('/content/data')
DATA = glob.glob('/content/data/*/metadata.csv')[0].rsplit('/', 1)[0]
VOICE = os.path.basename(DATA)
n = sum(1 for _ in open(DATA + '/metadata.csv', encoding='utf-8'))
print(f'Hlas: {VOICE}, vět: {n}')
if n < 300: print('⚠ Méně než 300 vět – hlas bude méně přesný; nahrajte víc, pokud to jde.')

In [ ]:
# 4) Výchozí bod: český hlas Jirka (dotrénování je mnohem rychlejší než trénovat od nuly)
from huggingface_hub import list_repo_files, hf_hub_download
REPO = 'rhasspy/piper-checkpoints'
files_ = [f for f in list_repo_files(REPO) if f.endswith('.ckpt')]
pick = [f for f in files_ if 'cs_CZ/jirka/medium' in f] or [f for f in files_ if 'en_US/lessac/medium' in f]
CKPT_START = hf_hub_download(REPO, pick[0], local_dir='/content/start')
print('Začínáme z:', pick[0])

In [ ]:
# 5) Trénování – průběh se ukládá na Disk; po odpojení stačí spustit znovu buňky 1–5 (pokračuje se od last.ckpt)
import glob
last = sorted(glob.glob(WORK + '/lightning_logs/*/checkpoints/last.ckpt'), key=os.path.getmtime)
# první spuštění: váhy z Jirky, vlastní běh od epochy 0 (warmstart); další spuštění: pokračování vlastního běhu
START = f'--ckpt_path "{last[-1]}"' if last else f'--model.warmstart_ckpt "{CKPT_START}"'
print('Pokračuji z', last[-1]) if last else print('Začínám z hlasu', CKPT_START)
EPOCHS = 2000   # doporučení projektu Piper pro dotrénování z méně než ~5 h nahrávek
!cd {WORK} && python3 -m piper.train fit \
  --data.voice_name "{VOICE}" \
  --data.csv_path "{DATA}/metadata.csv" \
  --data.audio_dir "{DATA}/wavs" \
  --model.sample_rate 22050 \
  --data.espeak_voice cs \
  --data.cache_dir /content/cache \
  --data.config_path "{WORK}/{VOICE}.onnx.json" \
  --data.batch_size 16 \
  --trainer.max_epochs {EPOCHS} \
  --trainer.default_root_dir "{WORK}" \
  {START}

In [ ]:
# 6) Export hlasu (.onnx + .onnx.json) a krátká ukázka
import glob, json
# nejnovější last.ckpt (projekt Piper doporučuje začít s ním; porovnat jde i val_mos / val_mel)
ck = sorted(glob.glob(WORK + '/lightning_logs/*/checkpoints/last.ckpt'), key=os.path.getmtime)[-1]
!python3 -m piper.train.export_onnx --checkpoint "{ck}" --output-file "{WORK}/{VOICE}.onnx"
cfg = json.load(open(f'{WORK}/{VOICE}.onnx.json', encoding='utf-8'))
cfg['dataset'] = VOICE
json.dump(cfg, open(f'{WORK}/{VOICE}.onnx.json', 'w', encoding='utf-8'), ensure_ascii=False)
!pip -q install piper-tts
!echo "Dobrý den, tohle je můj vlastní hlas v Překladači." | piper -m "{WORK}/{VOICE}.onnx" -f /content/ukazka.wav
from IPython.display import Audio
Audio('/content/ukazka.wav')

In [ ]:
# 7) Stažení – oba soubory dejte do jedné složky a v Překladači: Nastavení → Součásti → Importovat hlas (.onnx)
from google.colab import files
files.download(f'{WORK}/{VOICE}.onnx')
files.download(f'{WORK}/{VOICE}.onnx.json')